# Hands-on Modul 2.2: Safe & Efficient Continued Pretraining

Dalam hands-on ini, kita akan mensimulasikan skenario nyata seorang AI Engineer:
**"Bagaimana cara mengajarkan pengetahuan baru ke model tanpa membuatnya lupa pengetahuan lama, dan tanpa meledakkan memori GPU?"**

Kita akan menggabungkan dua teknik kunci:
1.  **Data Mixing (Replay):** Mencampur data baru dengan data lama.
2.  **LoRA (PEFT):** Melatih hanya adapter kecil, bukan seluruh model.

In [1]:
# Instalasi library ekosistem Hugging Face
# 'peft' adalah library kunci untuk LoRA
!pip install transformers datasets peft torch trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 11.9 MB/s eta 0:00:00


## Langkah 1: Strategi Data Mixing (Mencegah Forgetting)

Kita akan membuat dua dataset sintetis sederhana:
1.  **Domain Medis (Baru):** Teks yang ingin kita ajarkan.
2.  **Domain Umum (Replay):** Teks wikipedia/umum untuk menjaga ingatan model.

In [2]:
from datasets import Dataset, concatenate_datasets

# 1. Buat Data Domain Baru (Misal: 100 contoh teks medis)
medical_text = [
    "Hipertensi adalah kondisi tekanan darah tinggi kronis.",
    "Resep parasetamol dosis 500mg untuk demam ringan.",
    "Diagnosis diabetes tipe 2 membutuhkan tes HbA1c.",
    "Gejala flu meliputi batuk, pilek, dan nyeri tubuh."
] * 25 # Duplikasi agar jadi 100 baris

# 2. Buat Data Replay (Misal: 25 contoh teks umum - Rasio 4:1)
general_text = [
    "Ibu kota Prancis adalah Paris.",
    "Matahari adalah bintang di pusat tata surya.",
    "Pemrograman Python sangat populer untuk AI.",
    "Resep nasi goreng enak membutuhkan bumbu yang pas."
] * 7 # Duplikasi dikit

# 3. Konversi ke Hugging Face Dataset
ds_med = Dataset.from_dict({"text": medical_text})
ds_gen = Dataset.from_dict({"text": general_text})

print(f"Jumlah Data Medis: {len(ds_med)}")
print(f"Jumlah Data Replay: {len(ds_gen)}")

# 4. Lakukan Mixing & Shuffling (PENTING!)
# Kita gabungkan lalu acak agar model tidak bias urutan.
cpt_dataset = concatenate_datasets([ds_med, ds_gen]).shuffle(seed=42)

print(f"Total Data Training: {len(cpt_dataset)}")
print(f"Contoh Data: {cpt_dataset[0]}")

Jumlah Data Medis: 100
Jumlah Data Replay: 28
Total Data Training: 128
Contoh Data: {'text': 'Resep parasetamol dosis 500mg untuk demam ringan.'}


## Langkah 2: Setup Model & Tokenizer

Kita gunakan `gpt2` (model kecil) agar cepat, tapi teknik ini berlaku sama persis untuk Llama-3 atau Mistral.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

print("Memuat Tokenizer & Model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Fix umum untuk GPT-2

model = AutoModelForCausalLM.from_pretrained(model_name)

# Fungsi Tokenisasi
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = cpt_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Memuat Tokenizer & Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

## Langkah 3: Implementasi LoRA (Efisiensi Memori)

Di sini kita akan menggunakan library `peft` untuk menyuntikkan adapter LoRA. Perhatikan betapa sedikitnya parameter yang akan kita latih!

In [4]:
from peft import LoraConfig, get_peft_model, TaskType

# 1. Konfigurasi LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # Tipe tugas: Causal LM (Next token prediction)
    inference_mode=False,
    r=8,            # Rank: Ukuran matriks adapter (makin kecil makin hemat)
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1
)

# 2. Terapkan LoRA ke Model
model = get_peft_model(model, peft_config)

# 3. Lihat Penghematan Parameter
print("\n--- Statistik Parameter LoRA ---")
model.print_trainable_parameters()

# Output biasanya menunjukkan kita hanya melatih < 1% parameter!
# Ini berarti kita bisa melatih model besar di GPU kecil.


--- Statistik Parameter LoRA ---
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Langkah 4: Training dengan Safe Hyperparameters

Kita menggunakan `DataCollatorForLanguageModeling` (karena ini CPT/CLM, bukan SFT) dan *Learning Rate* yang rendah.

In [5]:
import torch #
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Collator untuk CLM (mlm=False)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Hyperparameters Aman untuk CPT
training_args = TrainingArguments(
    output_dir="./gpt2-medical-lora",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4, # Sedikit lebih tinggi karena LoRA, kalau Full CPT harus lebih rendah (2e-5)
    logging_steps=10,
    # Cek GPU secara otomatis
    use_cpu=False if torch.cuda.is_available() else True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("Mulai Training CPT (LoRA)...")
trainer.train()
print("Training Selesai!")

Mulai Training CPT (LoRA)...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,7.144070
20,6.895104
30,6.559803
40,6.459812


Training Selesai!


In [6]:
# --- Langkah 5: Tes Model Hasil CPT (LoRA) ---

# Kita ingin melihat apakah model sudah "menyerap" pola data medis kita.
# Ingat: Karena data kita sangat sedikit, model mungkin akan "menghafal" (overfitting).
# Tapi untuk tujuan demo teknik, ini justru membuktikan training BERHASIL mengubah bobot.

import torch

# 1. Siapkan Prompt (Potongan dari data latih medis kita)
prompt = "Resep parasetamol"

# 2. Tokenisasi
inputs = tokenizer(prompt, return_tensors="pt")

# Pindahkan input ke GPU jika ada (karena model ada di GPU setelah training)
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = {k: v.to(device) for k, v in inputs.items()}

# 3. Generate Output
print(f"Prompt: '{prompt}'")
print("Sedang generate jawaban...")

model.eval() # Set mode evaluasi
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=20, # Jangan terlalu panjang
        temperature=0.5,   # Rendah agar dia mengeluarkan apa yang dia hafal
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# 4. Decode dan Tampilkan
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("-" * 30)
print(f"Hasil Output:\n{generated_text}")
print("-" * 30)

print("Analisis: Jika output melanjutkan dengan 'dosis 500mg...' atau kalimat medis serupa,")
print("berarti LoRA adapter BERHASIL menyuntikkan pengetahuan baru ke GPT-2!")

Prompt: 'Resep parasetamol'
Sedang generate jawaban...
------------------------------
Hasil Output:
Resep parasetamol.

Tacquor, T. (1903). The effect of the presence of
------------------------------
Analisis: Jika output melanjutkan dengan 'dosis 500mg...' atau kalimat medis serupa,
berarti LoRA adapter BERHASIL menyuntikkan pengetahuan baru ke GPT-2!


# Mengapa hasilnya begini? Jika model berhasil melengkapi kalimat persis seperti data latih, itu artinya Transfer Knowledge Berhasil.

Di dunia nyata dengan dataset Terabyte, model tidak akan menghafal se-ekstrem ini, melainkan akan memahami konsep medisnya. Namun, mekanisme teknisnya (Adapter LoRA yang aktif dan mengubah probabilitas token) adalah sama persis.

### Kesimpulan dan Langkah Selanjutnya

Selamat! Anda telah berhasil melakukan simulasi **Continued Pretraining (CPT)** menggunakan teknik-teknik *engineering* modern.

**Apa yang telah Anda capai:**
1.  **Mitigasi Risiko:** Anda menerapkan **Data Mixing** (80% Medis + 20% Umum) untuk mencegah model "lupa" pengetahuan dasarnya (*Catastrophic Forgetting*).
2.  **Efisiensi Ekstrem:** Menggunakan **LoRA**, Anda menyuntikkan pengetahuan baru hanya dengan melatih **<1% parameter**, memungkinkan proses ini berjalan di GPU gratisan (T4) tanpa meledakkan memori.
3.  **Verifikasi:** Tes inferensi membuktikan bahwa model `gpt2` yang awalnya tidak tahu apa-apa tentang resep medis, kini mampu melengkapi kalimat dengan konteks yang benar (meskipun hasil *overfitting* pada data kecil ini).

** Tantangan Eksperimen (Try This!):**
Jangan berhenti di sini. Cobalah memodifikasi kode di atas:
* **Ganti Dataset:** Coba ganti teks medis dengan teks Hukum (UUD 1945) atau teks Koding (Python Documentation). Apakah model bisa beradaptasi?
* **Ganti Model:** Jika Anda punya akses GPU lebih besar (A100), coba ganti `model_name` menjadi `"meta-llama/Meta-Llama-3-8B"` (perlu login HuggingFace).
* **Ubah LoRA:** Coba ubah parameter `r` (rank) di konfigurasi LoRA dari 8 menjadi 16 atau 32. Apakah hasilnya lebih bagus?

Teknik inilah yang digunakan di industri untuk menciptakan model-model spesialis (seperti *Med-PaLM* atau *CodeLlama*) dengan biaya yang jauh lebih efisien daripada melatih dari nol.